In [3]:
%%capture
%uv add langchain==0.3.23 | tail -n 1 
%uv add openai==1.77.0 | tail -n 1
%uv add langchain-openai==0.3.16 | tail -n 1
%uv add langchain-community==0.3.16 | tail -n 1
%uv add wikipedia==1.4.0 | tail -n 1
%uv add python-dotenv | tail -n 1    

In [4]:
import os
from dotenv import load_dotenv

load_dotenv()
openAIKey = os.environ["OPENAI_API_KEY"]
print(f"OpenAI Key: {openAIKey[:10]}...{openAIKey[-10:]}")

OpenAI Key: sk-proj-5J...FleV4jNVUA


In [5]:
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(model_name="gpt-4.1-nano", openai_api_key=openAIKey, temperature=0)

### Function

In [6]:
def add_numbers(inputs:str) -> dict:
    """
    Adds a list of numbers provided in the input dictionary or extracts numbers from a string.

    Parameters:
    - inputs (str): 
    string, it should contain numbers that can be extracted and summed.

    Returns:
    - dict: A dictionary with a single key "result" containing the sum of the numbers.

    Example Input (Dictionary):
    {"numbers": [10, 20, 30]}

    Example Input (String):
    "Add the numbers 10, 20, and 30."

    Example Output:
    {"result": 60}
    """
    numbers = [int(x) for x in inputs.replace(",", "").split() if x.isdigit()]

    
    result = sum(numbers)
    return {"result": result}

In [12]:
add_numbers("10, 20, 30")

{'result': 60}

### Tool

In [16]:
from langchain.agents import Tool

add_tool = Tool(
    name="add_numbers",
    func=add_numbers,
    description="Add a list of numbers provided in the input string and return the result.."
)

print(add_tool)
print(add_tool.name)
print(add_tool.description)
print(add_tool.invoke)
print(add_tool.func)
print(add_tool.args_schema)
print(add_tool.func("Add the numbers 5, 15, and 25"))
print(add_tool.invoke("Add the numbers 10, 15, and 25"))

name='add_numbers' description='Add a list of numbers provided in the input string and return the result..' func=<function add_numbers at 0x117cecb80>
add_numbers
Add a list of numbers provided in the input string and return the result..
<bound method BaseTool.invoke of Tool(name='add_numbers', description='Add a list of numbers provided in the input string and return the result..', func=<function add_numbers at 0x117cecb80>)>
<function add_numbers at 0x117cecb80>
None
{'result': 45}
{'result': 50}


### @tool operator

In [19]:
from langchain_core.tools import tool
import re

@tool
def add_numbers(inputs:str) -> dict:
    """
    Adds a list of numbers provided in the input dictionary or extracts numbers from a string.

    Parameters:
    - inputs (str): 
    string, it should contain numbers that can be extracted and summed.

    Returns:
    - dict: A dictionary with a single key "result" containing the sum of the numbers.

    Example Input (Dictionary):
    {"numbers": [10, 20, 30]}

    Example Input (String):
    "Add the numbers 10, 20, and 30."

    Example Output:
    {"result": 60}
    """
    numbers = [int(num) for num in re.findall(r'\d+', inputs)]
    #numbers = [int(x) for x in inputs.replace(",", "").split() if x.isdigit()]

    
    result = sum(numbers)
    return {"result": result}

In [24]:
print(f"Name :\n {add_numbers.name}")
print(f"Description :\n {add_numbers.description}")
print(f"Args :\n {add_numbers.args}")
print(f"Has Schema: {hasattr(add_numbers, 'args_schema')}")

Name :
 add_numbers
Description :
 Adds a list of numbers provided in the input dictionary or extracts numbers from a string.

Parameters:
- inputs (str): 
string, it should contain numbers that can be extracted and summed.

Returns:
- dict: A dictionary with a single key "result" containing the sum of the numbers.

Example Input (Dictionary):
{"numbers": [10, 20, 30]}

Example Input (String):
"Add the numbers 10, 20, and 30."

Example Output:
{"result": 60}
Args :
 {'inputs': {'title': 'Inputs', 'type': 'string'}}
Has Schema: True


In [23]:
add_numbers.invoke("Add the numbers 10, 20, and 30.")

{'result': 60}

### @tool - Structured Tool

In [25]:
from typing import List

@tool
def add_numbers_with_options(numbers: List[float], absolute: bool = False) -> float:
    """
    Adds a list of numbers provided as input.

    Parameters:
    - numbers (List[float]): A list of numbers to be summed.
    - absolute (bool): If True, use the absolute values of the numbers before summing.

    Returns:
    - float: The total sum of the numbers.
    """
    if absolute:
        numbers = [abs(n) for n in numbers]
    return sum(numbers)

In [27]:
print(f"Has Attribute 'args_schema': {hasattr(add_numbers, 'args_schema')}")
print(f"Args :\n {add_numbers.args}")
print(f"Has Attribute 'args_schema': {hasattr(add_numbers_with_options, 'args_schema')}")
print(f"Args :\n {add_numbers_with_options.args}")

Has Attribute 'args_schema': True
Args :
 {'inputs': {'title': 'Inputs', 'type': 'string'}}
Has Attribute 'args_schema': True
Args :
 {'numbers': {'items': {'type': 'number'}, 'title': 'Numbers', 'type': 'array'}, 'absolute': {'default': False, 'title': 'Absolute', 'type': 'boolean'}}


In [28]:
print(add_numbers_with_options.invoke({"numbers":[-1.1,-2.1,-3.0],"absolute":False}))
print(add_numbers_with_options.invoke({"numbers":[-1.1,-2.1,-3.0],"absolute":True}))

-6.2
6.2


### Improved tool return type

In [29]:
@tool
def sum_numbers_from_text(inputs: str) -> float:
    """
    Adds a list of numbers provided in the input string.
    
    Args:
        text: A string containing numbers that should be extracted and summed.
        
    Returns:
        The sum of all numbers found in the input.
    """
    # Use regular expressions to extract all numbers from the input
    numbers = [int(num) for num in re.findall(r'\d+', inputs)]
    result = sum(numbers)
    return result

In [30]:
from typing import Dict, Union

@tool
def sum_numbers_with_complex_output(inputs: str) -> Dict[str, Union[float, str]]:
    """
    Extracts and sums all integers and decimal numbers from the input string.

    Parameters:
    - inputs (str): A string that may contain numeric values.

    Returns:
    - dict: A dictionary with the key "result". If numbers are found, the value is their sum (float). 
            If no numbers are found or an error occurs, the value is a corresponding message (str).

    Example Input:
    "Add 10, 20.5, and -3."

    Example Output:
    {"result": 27.5}
    """
    matches = re.findall(r'-?\d+(?:\.\d+)?', inputs)
    if not matches:
        return {"result": "No numbers found in input."}
    try:
        numbers = [float(num) for num in matches]
        total = sum(numbers)
        return {"result": total}
    except Exception as e:
        return {"result": f"Error during summation: {str(e)}"}

### LLM.run - May code keeps run indefintely

In [ ]:
from langchain.agents import initialize_agent

agent = initialize_agent([add_tool], llm, agent="zero-shot-react-description", verbose=True,  handle_parsing_errors=True, max_iterations=3 )
response = agent.run("In 2023, the US GDP was approximately $27.72 trillion, while Canada's was around $2.14 trillion and Mexico's was about $1.79 trillion what is the total.")
print(response)

/var/folders/wy/mmq3bg9s30s2zc02dq4r60z80000gn/T/ipykernel_17875/251587479.py:3: LangChainDeprecationWarning: LangChain agents will continue to be supported, but it is recommended for new use cases to be built with LangGraph. LangGraph offers a more flexible and full-featured framework for building agents, including support for tool-calling, persistence of state, and human-in-the-loop workflows. For details, refer to the `LangGraph documentation <https://langchain-ai.github.io/langgraph/>`_ as well as guides for `Migrating from AgentExecutor <https://python.langchain.com/docs/how_to/migrate_agent/>`_ and LangGraph's `Pre-built ReAct agent <https://langchain-ai.github.io/langgraph/how-tos/create-react-agent/>`_.
  agent = initialize_agent([add_tool], llm, agent="zero-shot-react-description", verbose=True, handle_parsing_errors=True)
/var/folders/wy/mmq3bg9s30s2zc02dq4r60z80000gn/T/ipykernel_17875/251587479.py:4: LangChainDeprecationWarning: The method `Chain.run` was deprecated in langc



> Entering new AgentExecutor chain...
Action: add_numbers
Action Input: "27.72, 2.14, 1.79"
Observation: {'result': 0}
Thought:Question: In 2023, the US GDP was approximately $27.72 trillion, while Canada's was around $2.14 trillion and Mexico's was about $1.79 trillion what is the total.
Thought: I need to add these three numbers to find the total GDP.
Action: add_numbers
Action Input: "27.72, 2.14, 1.79"
Observation: {'result': 0}
Thought:Question: In 2023, the US GDP was approximately $27.72 trillion, while Canada's was around $2.14 trillion and Mexico's was about $1.79 trillion what is the total.
Thought: I need to add these three numbers to find the total GDP.
Action: add_numbers
Action Input: "27.72, 2.14, 1.79"
Observation: {'result': 0}
Thought:Question: In 2023, the US GDP was approximately $27.72 trillion, while Canada's was around $2.14 trillion and Mexico's was about $1.79 trillion what is the total.
Thought: I need to add these three numbers to find the total GDP.
Action

In [33]:
agent_2 = initialize_agent([sum_numbers_from_text], llm, agent="structured-chat-zero-shot-react-description", 
                           verbose=True, handle_parsing_errors=True, max_iterations=5 )
response = agent_2.invoke({"input": "Add 10, 20 and 30"})
print(response)



> Entering new AgentExecutor chain...
{
  "action": "sum_numbers_from_text",
  "action_input": "10, 20 and 30"
}

> Finished chain.
{'input': 'Add 10, 20 and 30', 'output': '{\n  "action": "sum_numbers_from_text",\n  "action_input": "10, 20 and 30"\n}'}


### llm.invoke - Run the approprite tool

In [36]:
agent_3 = initialize_agent([sum_numbers_with_complex_output], llm, agent="openai-functions", 
                           verbose=True, handle_parsing_errors=True, max_iterations=3)
response = agent_3.invoke({"input": "Add 10, 20 and 30"})
print(response)



> Entering new AgentExecutor chain...

Invoking: `sum_numbers_with_complex_output` with `{'inputs': 'Add 10, 20 and 30'}`


{'result': 60.0}The sum of 10, 20, and 30 is 60.

> Finished chain.
{'input': 'Add 10, 20 and 30', 'output': 'The sum of 10, 20, and 30 is 60.'}


### structured-chat-zero-shot-react-description

In [44]:
from langchain_openai import ChatOpenAI

llm_ai = ChatOpenAI(model="gpt-4.1")

In [45]:
agent_2 = initialize_agent(
    [add_numbers_with_options],
    llm_ai,
    agent="structured-chat-zero-shot-react-description",
    verbose=True,
    max_iterations=5
)

response = agent_2.invoke({
    "input": "Add -10, -20, and -30 using absolute values."
})

print(response)



> Entering new AgentExecutor chain...
Action:
```
{
  "action": "add_numbers_with_options",
  "action_input": {
    "numbers": [-10, -20, -30],
    "absolute": true
  }
}
```

Observation: 60.0
Thought:{
  "action": "Final Answer",
  "action_input": "The sum of -10, -20, and -30 using their absolute values is 60."
}

> Finished chain.
{'input': 'Add -10, -20, and -30 using absolute values.', 'output': '{\n  "action": "Final Answer",\n  "action_input": "The sum of -10, -20, and -30 using their absolute values is 60."\n}'}


### openai-functions

In [47]:
agent_openai = initialize_agent(
    [add_numbers_with_options],
    llm_ai,
    agent="openai-functions",
    verbose=True
)

response = agent_openai.invoke({
    "input": "Add -10, -20, and -30 using absolute values."
})
print(response)



> Entering new AgentExecutor chain...

Invoking: `add_numbers_with_options` with `{'numbers': [-10, -20, -30], 'absolute': True}`


60.0When adding -10, -20, and -30 using their absolute values, the result is 60.

> Finished chain.
{'input': 'Add -10, -20, and -30 using absolute values.', 'output': 'When adding -10, -20, and -30 using their absolute values, the result is 60.'}


### create_react_agent

In [80]:
#%%capture
!uv pip install langgraph | tail -n 1

Audited 1 package in 17ms


In [82]:
from langgraph.prebuilt import create_react_agent

agent_exec = create_react_agent(model=llm, tools=[sum_numbers_from_text])
msgs = agent_exec.invoke({"messages": [("human", "Add the numbers -10, -20, -30")]})
print(msgs["messages"][-1].content)

/var/folders/wy/mmq3bg9s30s2zc02dq4r60z80000gn/T/ipykernel_17875/1660207878.py:3: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  agent_exec = create_react_agent(model=llm, tools=[sum_numbers_from_text])


The sum of the numbers -10, -20, and -30 is 60.


### Subtraction Tool

In [84]:
@tool
def subtract_numbers(inputs: str) -> dict:
    """
    Extracts numbers from a string, negates the first number, and successively subtracts 
    the remaining numbers in the list.

    This function is designed to handle input in string format, where numbers are separated 
    by spaces, commas, or other delimiters. It parses the string, extracts valid numeric values, 
    and performs a step-by-step subtraction operation starting with the first number negated.

    Parameters:
    - inputs (str): 
      A string containing numbers to subtract. The string may include spaces, commas, or 
      other delimiters between the numbers.

    Returns:
    - dict: 
      A dictionary containing the key "result" with the calculated difference as its value. 
      If no valid numbers are found in the input string, the result defaults to 0.

    Example Input:
    "100, 20, 10"

    Example Output:
    {"result": -130}

    Notes:
    - Non-numeric characters in the input are ignored.
    - If the input string contains only one valid number, the result will be that number negated.
    - Handles a variety of delimiters (e.g., spaces, commas) but does not validate input formats 
      beyond extracting numeric values.
    """
    # Extract numbers from the string
    numbers = [int(num) for num in inputs.replace(",", "").split() if num.isdigit()]

    # If no numbers are found, return 0
    if not numbers:
        return {"result": 0}

    # Start with the first number negated
    result = -1 * numbers[0]

    # Subtract all subsequent numbers
    for num in numbers[1:]:
        result -= num

    return {"result": result}

In [85]:
print("Calling Tool Function:")
test_input = "10 20 30 and four a b" 
print(subtract_numbers.invoke(test_input))  # Example

Calling Tool Function:
{'result': -60}


In [86]:
# Multiplication Tool
@tool
def multiply_numbers(inputs: str) -> dict:
    """
    Extracts numbers from a string and calculates their product.

    Parameters:
    - inputs (str): A string containing numbers separated by spaces, commas, or other delimiters.

    Returns:
    - dict: A dictionary with the key "result" containing the product of the numbers.

    Example Input:
    "2, 3, 4"

    Example Output:
    {"result": 24}

    Notes:
    - If no numbers are found, the result defaults to 1 (neutral element for multiplication).
    """
    # Extract numbers from the string
    numbers = [int(num) for num in inputs.replace(",", "").split() if num.isdigit()]
    print(numbers)

    # If no numbers are found, return 1
    if not numbers:
        return {"result": 1}

    # Calculate the product of the numbers
    result = 1
    for num in numbers:
        result *= num
        print(num)

    return {"result": result}

In [87]:
# Division Tool
@tool
def divide_numbers(inputs: str) -> dict:
    """
    Extracts numbers from a string and calculates the result of dividing the first number 
    by the subsequent numbers in sequence.

    Parameters:
    - inputs (str): A string containing numbers separated by spaces, commas, or other delimiters.

    Returns:
    - dict: A dictionary with the key "result" containing the quotient.

    Example Input:
    "100, 5, 2"

    Example Output:
    {"result": 10.0}

    Notes:
    - If no numbers are found, the result defaults to 0.
    - Division by zero will raise an error.
    """
    # Extract numbers from the string
    numbers = [int(num) for num in inputs.replace(",", "").split() if num.isdigit()]


    # If no numbers are found, return 0
    if not numbers:
        return {"result": 0}

    # Calculate the result of dividing the first number by subsequent numbers
    result = numbers[0]
    for num in numbers[1:]:
        result /= num

    return {"result": result}

In [88]:
# Testing multiply_tool
multiply_test_input = "2, 3, and four "
multiply_result = multiply_numbers.invoke(multiply_test_input)
print("--- Testing MultiplyTool ---")
print(f"Input: {multiply_test_input}")
print(f"Output: {multiply_result}")

[2, 3]
2
3
--- Testing MultiplyTool ---
Input: 2, 3, and four 
Output: {'result': 6}


In [91]:
# Testing divide_tool
divide_test_input = "100, 10, two"
divide_result = divide_numbers.invoke(divide_test_input)
print("--- Testing DivideTool ---")
print(f"Input: {divide_test_input}")
print(f"Output: {divide_result}")

--- Testing DivideTool ---
Input: 100, 10, two
Output: {'result': 10.0}


In [92]:
tools = [add_numbers,subtract_numbers, multiply_numbers, divide_numbers]
tools

[StructuredTool(name='add_numbers', description='Adds a list of numbers provided in the input dictionary or extracts numbers from a string.\n\nParameters:\n- inputs (str): \nstring, it should contain numbers that can be extracted and summed.\n\nReturns:\n- dict: A dictionary with a single key "result" containing the sum of the numbers.\n\nExample Input (Dictionary):\n{"numbers": [10, 20, 30]}\n\nExample Input (String):\n"Add the numbers 10, 20, and 30."\n\nExample Output:\n{"result": 60}', args_schema=<class 'langchain_core.utils.pydantic.add_numbers'>, func=<function add_numbers at 0x122349bc0>),
 StructuredTool(name='subtract_numbers', description='Extracts numbers from a string, negates the first number, and successively subtracts \nthe remaining numbers in the list.\n\nThis function is designed to handle input in string format, where numbers are separated \nby spaces, commas, or other delimiters. It parses the string, extracts valid numeric values, \nand performs a step-by-step sub

In [93]:
from langgraph.prebuilt import create_react_agent

# Create the agent with all tools
math_agent = create_react_agent(
    model=llm,
    tools=tools,
    # Optional: Add a system message to guide the agent's behavior
    prompt="You are a helpful mathematical assistant that can perform various operations. Use the tools precisely and explain your reasoning clearly."
)

/var/folders/wy/mmq3bg9s30s2zc02dq4r60z80000gn/T/ipykernel_17875/1356249252.py:4: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  math_agent = create_react_agent(


In [94]:
response = math_agent.invoke({
    "messages": [("human", "What is 25 divided by 4?")]
})

# Get the final answer
final_answer = response["messages"][-1].content
print(final_answer)

25 divided by 4 is 6.25.


In [95]:
response_2 = math_agent.invoke({
    "messages": [("human", "Subtract 100, 20, and 10.")]
})

# Get the final answer
final_answer_2 = response_2["messages"][-2].content
print(final_answer_2)

{"result": -130}


In [96]:
print("\n--- Testing MultiplyTool ---")
response = math_agent.invoke({
    "messages": [("human", "Multiply 2, 3, and four.")]
})
print("Agent Response:", response["messages"][-1].content)

print("\n--- Testing DivideTool ---")
response = math_agent.invoke({
    "messages": [("human", "Divide 100 by 5 and then by 2.")]
})
print("Agent Response:", response["messages"][-1].content)


--- Testing MultiplyTool ---
[2, 3, 4]
2
3
4
[2, 3, 4]
2
3
4
Agent Response: The product of 2, 3, and 4 is 24.

--- Testing DivideTool ---
Agent Response: Dividing 100 by 5 gives 20. Then, dividing 20 by 2 gives 10.


In [97]:
@tool
def new_subtract_numbers(inputs: str) -> dict:
    """
    Extracts numbers from a string and performs subtraction sequentially, starting with the first number.

    This function is designed to handle input in string format, where numbers may be separated by spaces, 
    commas, or other delimiters. It parses the input string, extracts numeric values, and calculates 
    the result by subtracting each subsequent number from the first. inputs[0]-inputs[1]-inputs[2]

    Parameters:
    - inputs (str): 
      A string containing numbers to subtract. The string can include spaces, commas, or other 
      delimiters between the numbers.

    Returns:
    - dict: 
      A dictionary containing the key "result" with the calculated difference as its value. 
      If no valid numbers are found in the input string, the result defaults to 0.

    Example Usage:
    - Input: "100, 20, 10"
    - Output: {"result": 70}

    Limitations:
    - The function does not handle cases where numbers are formatted with decimals or other non-integer representations.
    """
    # Extract numbers from the string
    numbers = [int(num) for num in inputs.replace(",", "").split() if num.isdigit()]

    # If no numbers are found, return 0
    if not numbers:
        return {"result": 0}

    # Start with the first number
    result = numbers[0]

    # Subtract all subsequent numbers
    for num in numbers[1:]:
        result -= num

    return {"result": result}

In [102]:
tools_updated = [add_numbers, new_subtract_numbers, multiply_numbers, divide_numbers]
# Create the agent with all tools
math_agent_new = create_react_agent(
    model=llm_ai,
    tools=tools_updated,
    # Optional: Add a system message to guide the agent's behavior
    prompt="You are a helpful mathematical assistant that can perform various operations. Use the tools precisely and explain your reasoning clearly."
)
print("agent",math_agent_new)

agent <langgraph.graph.state.CompiledStateGraph object at 0x126bb0e10>


/var/folders/wy/mmq3bg9s30s2zc02dq4r60z80000gn/T/ipykernel_17875/1976245343.py:3: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  math_agent_new = create_react_agent(


In [103]:
# Test Cases
test_cases = [
    {
        "query": "Subtract 100, 20, and 10.",
        "expected": {"result": 70},
        "description": "Testing subtraction tool with sequential subtraction."
    },
    {
        "query": "Multiply 2, 3, and 4.",
        "expected": {"result": 24},
        "description": "Testing multiplication tool for a list of numbers."
    },
    {
        "query": "Divide 100 by 5 and then by 2.",
        "expected": {"result": 10.0},
        "description": "Testing division tool with sequential division."
    },
    {
        "query": "Subtract 50 from 20.",
        "expected": {"result": -30},
        "description": "Testing subtraction tool with negative results."
    }

]

In [104]:
correct_tasks = []
# Corrected test execution
for index, test in enumerate(test_cases, start=1):
    query = test["query"]
    expected_result = test["expected"]["result"]  # Extract just the value
    
    print(f"\n--- Test Case {index}: {test['description']} ---")
    print(f"Query: {query}")
    
    # Properly format the input
    response = math_agent_new.invoke({"messages": [("human", query)]})
    
    # Find the tool message in the response
    tool_message = None
    for msg in response["messages"]:
        if hasattr(msg, 'name') and msg.name in ['add_numbers', 'new_subtract_numbers', 'multiply_numbers', 'divide_numbers']:
            tool_message = msg
            break
    
    if tool_message:
        # Parse the tool result from its content
        import json
        tool_result = json.loads(tool_message.content)["result"]
        print(f"Tool Result: {tool_result}")
        print(f"Expected Result: {expected_result}")
        
        if tool_result == expected_result:
            print(f"✅ Test Passed: {test['description']}")
            correct_tasks.append(test["description"])
        else:
            print(f"❌ Test Failed: {test['description']}")
    else:
        print("❌ No tool was called by the agent")

print("\nCorrectly passed tests:", correct_tasks)


--- Test Case 1: Testing subtraction tool with sequential subtraction. ---
Query: Subtract 100, 20, and 10.
Tool Result: 70
Expected Result: 70
✅ Test Passed: Testing subtraction tool with sequential subtraction.

--- Test Case 2: Testing multiplication tool for a list of numbers. ---
Query: Multiply 2, 3, and 4.
[2, 3, 4]
2
3
4
Tool Result: 24
Expected Result: 24
✅ Test Passed: Testing multiplication tool for a list of numbers.

--- Test Case 3: Testing division tool with sequential division. ---
Query: Divide 100 by 5 and then by 2.
Tool Result: 10.0
Expected Result: 10.0
✅ Test Passed: Testing division tool with sequential division.

--- Test Case 4: Testing subtraction tool with negative results. ---
Query: Subtract 50 from 20.
Tool Result: -30
Expected Result: -30
✅ Test Passed: Testing subtraction tool with negative results.

Correctly passed tests: ['Testing subtraction tool with sequential subtraction.', 'Testing multiplication tool for a list of numbers.', 'Testing division t

### langchain built-in tools

In [111]:
!uv pip install langchain-community==0.3.16 | tail -n 1
!uv pip install wikipedia | tail -n 1

Audited 1 package in 20ms
Resolved 9 packages in 661ms                                         
   Building wikipedia==1.4.0                                           
   Building wikipedia==1.4.0                                   
⠙ Preparing packages... (0/3)
   Building wikipedia==1.4.0------     0 B/103.90 KiB          
⠙ Preparing packages... (0/3)
   Building wikipedia==1.4.0------ 14.84 KiB/103.90 KiB        
⠙ Preparing packages... (0/3)
   Building wikipedia==1.4.0------ 14.84 KiB/103.90 KiB        
⠙ Preparing packages... (0/3)
soupsieve            ------------------------------     0 B/35.82 KiB
   Building wikipedia==1.4.0------ 14.84 KiB/103.90 KiB        
⠙ Preparing packages... (0/3)
soupsieve            ------------------------------ 14.88 KiB/35.82 KiB
   Building wikipedia==1.4.0------ 14.84 KiB/103.90 KiB        
⠙ Preparing packages... (0/3)
soupsieve            ------------------------------ 14.88 KiB/35.82 KiB
   Building wikipedia==1.4.0------ 30.84 KiB/103.90 Ki

### Using the Wikipedia tool

In [113]:
from langchain_community.utilities import WikipediaAPIWrapper

# Create a Wikipedia tool using the @tool decorator
@tool
def search_wikipedia(query: str) -> str:
    """Search Wikipedia for factual information about a topic.
    
    Parameters:
    - query (str): The topic or question to search for on Wikipedia
    
    Returns:
    - str: A summary of relevant information from Wikipedia
    """
    wiki = WikipediaAPIWrapper()
    return wiki.run(query)

In [114]:
search_wikipedia.invoke("What is tool calling?")

'Page: Cold calling\nSummary: Cold calling is the solicitation of business from potential customers who have had no prior contact with the salesperson conducting the call. It is an attempt to convince potential customers to purchase the salesperson\'s product or service.  Generally, it is an over-the-phone process, making it a form of telemarketing, but can also be done in-person by door-to-door salespeople.  Though cold calling can be used as a legitimate business tool, scammers can use cold calling as well.\n\nPage: What Is a Woman?\nSummary: What Is a Woman? is a 2022 American documentary film about gender and transgender issues, directed by Justin Folk and presented by conservative political commentator Matt Walsh. The film was released by conservative website The Daily Wire. In the film, Walsh asks various people "What is a woman?" with the goal of showing them that their definition of womanhood is circular. Walsh said he made the film in opposition to gender ideology. It is descr

In [115]:
tools_updated = [add_numbers, new_subtract_numbers, multiply_numbers, divide_numbers, search_wikipedia]

# Create the agent with all tools including Wikipedia
math_agent_updated = create_react_agent(
    model=llm,
    tools=tools_updated,
    prompt="You are a helpful assistant that can perform various mathematical operations and look up information. Use the tools precisely and explain your reasoning clearly."
)

/var/folders/wy/mmq3bg9s30s2zc02dq4r60z80000gn/T/ipykernel_17875/2888925466.py:4: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  math_agent_updated = create_react_agent(


In [116]:
query = "What is the population of Canada? Multiply it by 0.75"

response = math_agent_updated.invoke({"messages": [("human", query)]})

print("\nMessage sequence:")
for i, msg in enumerate(response["messages"]):
    print(f"\n--- Message {i+1} ---")
    print(f"Type: {type(msg).__name__}")
    if hasattr(msg, 'content'):
        print(f"Content: {msg.content}")
    if hasattr(msg, 'name'):
        print(f"Name: {msg.name}")
    if hasattr(msg, 'tool_calls') and msg.tool_calls:
        print(f"Tool calls: {msg.tool_calls}")

[]

Message sequence:

--- Message 1 ---
Type: HumanMessage
Content: What is the population of Canada? Multiply it by 0.75
Name: None

--- Message 2 ---
Type: AIMessage
Content: 
Name: None
Tool calls: [{'name': 'search_wikipedia', 'args': {'query': 'Population of Canada'}, 'id': 'call_94Oj0vbn7a6wCgWPnpOiDIL6', 'type': 'tool_call'}]

--- Message 3 ---
Type: ToolMessage
Content: Page: Population of Canada
Summary: Canada ranks 37th by population among countries of the world, comprising about 0.5% of the world's total, with about 41.5 million Canadians as of 2025. Despite being the second-largest country by total area (fourth-largest by land area), the vast majority of the country is sparsely inhabited, with most of its population south of the 55th parallel north. Just over 60 percent of Canadians live in just two provinces: Ontario and Quebec. Though Canada's overall population density is low, many regions in the south, such as the Quebec City–Windsor Corridor, have population densitie

### Create a power tool to calculate exponents

In [117]:
def calculate_power(input_text: str) -> dict:
    """
    Calculates the power of a number (x^y).

    Parameters:
    - input_text (str): A string like "2, 3", "2 3", "5^2", or "2 to the power of 3".

    Returns:
    - dict: {"result": <calculated value>} or an error message.
    """
    # Try to extract expressions like "5^2"
    match = re.search(r"(\d+(?:\.\d+)?)\s*\^+\s*(\d+(?:\.\d+)?)", input_text)
    if match:
        base = float(match.group(1))
        exponent = float(match.group(2))
        return {"result": base ** exponent}

    # Try to extract expressions like "2 to the power of 3"
    match = re.search(r"(\d+(?:\.\d+)?)\s*(?:to\s+the\s+power\s+of)\s*(\d+(?:\.\d+)?)", input_text, re.IGNORECASE)
    if match:
        base = float(match.group(1))
        exponent = float(match.group(2))
        return {"result": base ** exponent}

    # Fallback: assume two numbers separated by space or comma
    try:
        numbers = [float(num) for num in input_text.replace(",", " ").split()]
        if len(numbers) != 2:
            return {"result": "Invalid input. Please provide exactly two numbers."}
        base, exponent = numbers
        return {"result": base ** exponent}
    except ValueError:
        return {"result": "Invalid input format. Provide input like '2 3', '2^3', or '2 to the power of 3'."}

In [118]:
power_tool = Tool(
   name="PowerTool",
   func=calculate_power,
   description="Calculates the power of a number (x^y). Input should be two numbers: base and exponent."
)

In [120]:
# List of tools for the agent
tools = [power_tool]

# Create the agent
agent = initialize_agent(
   tools,
   llm,
   agent="zero-shot-react-description",
   verbose=True,
    handle_parsing_errors=True
)

agent.run("Calculate 5 to the power of 2.")



> Entering new AgentExecutor chain...
Action: PowerTool
Action Input: 5, 2
Observation: {'result': 25.0}
Thought:Thought: I now know the final answer
Final Answer: 25

> Finished chain.


'25'